# 03 - Feature Engineering

Stage 3 of the pipeline. Builds the master feature matrix that the modelling layer consumes.

**Inputs**
- `outputs/gdelt_preprocessed.csv` from 01_GDELT_EDA
- `outputs/icews_weekly_all.csv` from 02_ICEWS_EDA

**Output**
- `outputs/features_master.csv` and `.parquet` - one row per (corridor, week) with raw, rolling, and lag features.

**Steps (matching the Stage 3 diagram)**
1. Collapse GDELT and ICEWS to per-corridor weekly resolution (both directions combined).
2. Build the master weekly spine and merge both datasets.
3. Forward-fill structural gaps within each corridor.
4. Rolling means at 4-week and 8-week windows per feature.
5. Rolling volatility (std) at 4-week window per feature.
6. Lag features at T-4, T-8, T-13 weeks (approx 30, 60, 90 day prediction horizons).
7. Save the matrix, ready for NVS data integration in Stage 4.

In [1]:
# Cell 1: Imports and corridor configuration

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

# 13 NVS corridors - same set used in master validation (02_ICEWS_EDA)
# Stored as (GDELT_actor1, GDELT_actor2). Direction is irrelevant at the
# feature-engineering stage; signals are aggregated across both directions.
CORRIDORS_GDELT = [
    ('USA', 'CHN'), ('USA', 'DEU'), ('USA', 'IND'), ('USA', 'BRA'),
    ('USA', 'CAN'), ('USA', 'MEX'), ('CHN', 'DEU'), ('CHN', 'IND'),
    ('CHN', 'JPN'), ('CHN', 'KOR'), ('DEU', 'IND'), ('DEU', 'BRA'),
    ('IND', 'BRA'),
]

# Mapping GDELT ISO3 codes -> ICEWS full country names
GDELT_TO_ICEWS = {
    'USA': 'United States', 'CHN': 'China',       'DEU': 'Germany',
    'FRA': 'France',        'IND': 'India',       'BRA': 'Brazil',
    'CAN': 'Canada',        'MEX': 'Mexico',      'JPN': 'Japan',
    'KOR': 'South Korea',   'DNK': 'Denmark',
}

def canonical_pair(a, b):
    """Direction-agnostic corridor key from two country codes."""
    return '_'.join(sorted([a, b]))

CORRIDOR_KEYS = sorted({canonical_pair(a, b) for a, b in CORRIDORS_GDELT})

print(f'Corridors in scope: {len(CORRIDOR_KEYS)}')
for k in CORRIDOR_KEYS:
    print(f'  {k}')

Corridors in scope: 13
  BRA_DEU
  BRA_IND
  BRA_USA
  CAN_USA
  CHN_DEU
  CHN_IND
  CHN_JPN
  CHN_KOR
  CHN_USA
  DEU_IND
  DEU_USA
  IND_USA
  MEX_USA


### Why collapse both directions

GDELT and ICEWS record events as directed (`actor1 -> actor2`). For tone, hostility, and intensity, the two directions are essentially mirror images of the same bilateral relationship. Keeping them separate would double the column count without adding signal.

Tariff direction does matter - but that's a label dimension added at **Stage 4** when the NVS internal trade matrix arrives. At the geopolitical-signal level (this stage), undirected corridor pairs are sufficient.

In [2]:
# Cell 2: Load GDELT and collapse to per-corridor weekly

gdelt = pd.read_csv('../outputs/gdelt_preprocessed.csv')
gdelt['week_start'] = pd.to_datetime(gdelt['week_start'])

# Tag canonical corridor and filter to the 13
gdelt['corridor'] = [
    canonical_pair(a, b) for a, b in zip(gdelt['actor1'], gdelt['actor2'])
]
gdelt_13 = gdelt[gdelt['corridor'].isin(CORRIDOR_KEYS)].copy()

# Aggregate both directions per (corridor, week)
# Counts/sums are summed across directions, rates/means are averaged
GDELT_AGG = {
    'all_event_count':           'sum',
    'all_avg_goldstein':         'mean',
    'all_min_goldstein':         'min',
    'all_avg_tone':              'mean',
    'all_total_articles':        'sum',
    'all_total_mentions':        'sum',
    'trade_event_count':         'sum',
    'trade_avg_tone':            'mean',
    'trade_total_articles':      'sum',
    'cooperative_event_count':   'sum',
    'hostile_event_count':       'sum',
    'hostility_ratio':           'mean',
    'trade_event_ratio':         'mean',
    'log_total_articles':        'mean',
    'log_trade_articles':        'mean',
    'tension_score':             'mean',
    'log_trade_event_count':     'mean',
    'log_tension_score_abs':     'mean',
    'trade_event_flag':          'max',
    'high_trade_activity':       'max',
    'hostility_ratio_norm':      'mean',
}

gdelt_corr = gdelt_13.groupby(
    ['corridor', 'week_start'], as_index=False
).agg(GDELT_AGG)

print(f'GDELT input rows:         {len(gdelt):,}')
print(f'GDELT 13-corridor rows:   {len(gdelt_13):,}')
print(f'After direction collapse: {len(gdelt_corr):,}')
print(f'Distinct weeks:           {gdelt_corr["week_start"].nunique()}')
print(f'Date range:               {gdelt_corr["week_start"].min().date()} to {gdelt_corr["week_start"].max().date()}')
print()
print('=== Rows per corridor ===')
print(gdelt_corr.groupby('corridor').size().to_string())

GDELT input rows:         64,395
GDELT 13-corridor rows:   12,984
After direction collapse: 6,545
Distinct weeks:           510
Date range:               2015-03-30 to 2024-12-30

=== Rows per corridor ===
corridor
BRA_DEU    508
BRA_IND    429
BRA_USA    510
CAN_USA    510
CHN_DEU    510
CHN_IND    510
CHN_JPN    510
CHN_KOR    510
CHN_USA    510
DEU_IND    508
DEU_USA    510
IND_USA    510
MEX_USA    510


In [3]:
# Cell 3: Load ICEWS and collapse to per-corridor weekly

icews = pd.read_csv('../outputs/icews_weekly_all.csv')
icews['week_start'] = pd.to_datetime(icews['week_start'])

# Map ICEWS country names back to GDELT codes
ICEWS_TO_GDELT = {v: k for k, v in GDELT_TO_ICEWS.items()}
icews['actor1'] = icews['source_country'].map(ICEWS_TO_GDELT)
icews['actor2'] = icews['target_country'].map(ICEWS_TO_GDELT)
icews = icews.dropna(subset=['actor1', 'actor2'])

icews['corridor'] = [
    canonical_pair(a, b) for a, b in zip(icews['actor1'], icews['actor2'])
]
icews_13 = icews[icews['corridor'].isin(CORRIDOR_KEYS)].copy()

# Aggregate both directions per (corridor, week)
icews_corr = icews_13.groupby(['corridor', 'week_start'], as_index=False).agg(
    icews_event_count=('event_count', 'sum'),
    icews_avg_intensity=('avg_intensity', 'mean'),
    icews_min_intensity=('min_intensity', 'min'),
    icews_trade_event_count=('trade_event_count', 'sum'),
    gdelt_confidence=('gdelt_confidence', 'first'),
)

print(f'ICEWS input rows:         {len(icews):,}')
print(f'ICEWS 13-corridor rows:   {len(icews_13):,}')
print(f'After direction collapse: {len(icews_corr):,}')
print(f'Distinct weeks:           {icews_corr["week_start"].nunique()}')
print(f'Date range:               {icews_corr["week_start"].min().date()} to {icews_corr["week_start"].max().date()}')
print()
print('=== Confidence per corridor ===')
print(icews_corr.groupby('corridor')['gdelt_confidence'].first().to_string())

ICEWS input rows:         19,508
ICEWS 13-corridor rows:   8,293
After direction collapse: 4,369
Distinct weeks:           417
Date range:               2014-12-29 to 2022-12-26

=== Confidence per corridor ===
corridor
BRA_DEU    moderate
BRA_IND         low
BRA_USA         low
CAN_USA         low
CHN_DEU    moderate
CHN_IND    moderate
CHN_JPN    moderate
CHN_KOR    moderate
CHN_USA        high
DEU_IND         low
DEU_USA    moderate
IND_USA         low
MEX_USA    moderate


### Master weekly spine

The spine is the cartesian product of `13 corridors x N Monday-weeks`. This guarantees every corridor has a row for every week, even where neither dataset had events to record.

Two reasons for an explicit spine rather than relying on the union of input rows:

1. **Rolling and lag operations assume regular time spacing.** If a corridor is missing whole weeks, a rolling window misaligns.
2. **Downstream the NVS data will arrive at irregular cadence.** Joining onto a regular weekly spine is cheap; reshaping later is not.

Forward-fill carries the last known value forward within each corridor for numeric features. This is a conservative assumption: "if nothing was reported this week, the bilateral situation is the same as last week." For binary flags (`trade_event_flag`) the forward-fill is benign because zero is the natural default and the source data already encodes it correctly.

In [4]:
# Cell 4: Build master weekly spine and merge GDELT + ICEWS

def snap_to_monday(s):
    return s - pd.to_timedelta(s.dt.weekday, unit='D')

# Snap both sources to Monday (defensive - they should already be)
gdelt_corr['week_start'] = snap_to_monday(gdelt_corr['week_start'])
icews_corr['week_start'] = snap_to_monday(icews_corr['week_start'])

# Re-aggregate in case any rows collapsed after the snap
gdelt_corr = gdelt_corr.groupby(['corridor', 'week_start'], as_index=False).agg(GDELT_AGG)
icews_corr = icews_corr.groupby(['corridor', 'week_start'], as_index=False).agg({
    'icews_event_count':       'sum',
    'icews_avg_intensity':     'mean',
    'icews_min_intensity':     'min',
    'icews_trade_event_count': 'sum',
    'gdelt_confidence':        'first',
})

# Master grid: every (corridor, Monday) combination bounded by GDELT range
all_weeks = sorted(gdelt_corr['week_start'].unique())
print(f'Master grid: {len(CORRIDOR_KEYS)} corridors x {len(all_weeks)} weeks = {len(CORRIDOR_KEYS)*len(all_weeks):,} rows')

spine = pd.MultiIndex.from_product(
    [CORRIDOR_KEYS, all_weeks],
    names=['corridor', 'week_start']
).to_frame(index=False)

# Merge
master = spine.merge(gdelt_corr, on=['corridor', 'week_start'], how='left')
master = master.merge(icews_corr, on=['corridor', 'week_start'], how='left')

# Confidence flag is constant per corridor - propagate across the corridor
master['gdelt_confidence'] = (
    master.groupby('corridor')['gdelt_confidence']
    .transform(lambda s: s.dropna().iloc[0] if s.dropna().size else 'unvalidated')
)

# Forward-fill numeric features within each corridor
num_cols = [c for c in master.columns
            if c not in ('corridor', 'week_start', 'gdelt_confidence')]
master = master.sort_values(['corridor', 'week_start']).reset_index(drop=True)
master[num_cols] = master.groupby('corridor')[num_cols].ffill()

print(f'Master spine shape:       {master.shape}')
print(f'  Corridors:              {master["corridor"].nunique()}')
print(f'  Weeks per corridor:     {len(master) // master["corridor"].nunique()}')
print()
print('=== Top missing-value columns after forward-fill ===')
nulls = master.isnull().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)
print(nulls.head(15).to_string() if len(nulls) else '(no nulls)')

Master grid: 13 corridors x 510 weeks = 6,630 rows
Master spine shape:       (6630, 28)
  Corridors:              13
  Weeks per corridor:     510

=== Top missing-value columns after forward-fill ===
icews_event_count          19
icews_avg_intensity        19
icews_min_intensity        19
icews_trade_event_count    19


### Rolling features

Two windows, picked to match the prediction horizons defined later:

- **4-week mean (`_ma4`)** - smooths out single-week noise. Roughly "the current month".
- **8-week mean (`_ma8`)** - captures slower regime shifts. Roughly "the current quarter".
- **4-week std (`_std4`)** - rolling volatility. Spikes when the relationship destabilises, even if the average tone has not moved much yet. Often a stronger early-warning signal than the mean alone for regime-change detection.

Binary trade-event flags get rolling **sums** rather than means: the count of trade-event weeks in the past 4 / 8 weeks. A model can use this as "tariff activity recency" without dealing with the structurally sparse raw flag.

In [5]:
# Cell 5: Rolling means - 4-week and 8-week per continuous feature

ROLLING_FEATURES = [
    'all_avg_tone',
    'all_avg_goldstein',
    'all_min_goldstein',
    'hostility_ratio',
    'hostility_ratio_norm',
    'tension_score',
    'trade_avg_tone',
    'trade_event_ratio',
    'log_total_articles',
    'log_trade_articles',
    'log_trade_event_count',
    'log_tension_score_abs',
    'icews_avg_intensity',
    'icews_min_intensity',
]
ROLLING_FEATURES = [c for c in ROLLING_FEATURES if c in master.columns]

for col in ROLLING_FEATURES:
    master[f'{col}_ma4'] = master.groupby('corridor')[col].transform(
        lambda s: s.rolling(window=4, min_periods=2).mean()
    )
    master[f'{col}_ma8'] = master.groupby('corridor')[col].transform(
        lambda s: s.rolling(window=8, min_periods=4).mean()
    )

print(f'Added {2 * len(ROLLING_FEATURES)} rolling-mean columns '
      f'({len(ROLLING_FEATURES)} features x 2 windows)')
print(f'Master shape now: {master.shape}')

Added 28 rolling-mean columns (14 features x 2 windows)
Master shape now: (6630, 56)


In [6]:
# Cell 6: Rolling volatility - 4-week std per continuous feature

VOLATILITY_FEATURES = ROLLING_FEATURES  # same set as the means

for col in VOLATILITY_FEATURES:
    master[f'{col}_std4'] = master.groupby('corridor')[col].transform(
        lambda s: s.rolling(window=4, min_periods=2).std()
    )

print(f'Added {len(VOLATILITY_FEATURES)} rolling-std columns')
print(f'Master shape now: {master.shape}')

Added 14 rolling-std columns
Master shape now: (6630, 70)


In [7]:
# Cell 7: Rolling sums for binary event flags
# Counts how many trade-event weeks occurred in the past 4 / 8 weeks

BINARY_FEATURES = [c for c in ('trade_event_flag', 'high_trade_activity')
                   if c in master.columns]

for col in BINARY_FEATURES:
    master[f'{col}_sum4'] = master.groupby('corridor')[col].transform(
        lambda s: s.rolling(window=4, min_periods=1).sum()
    )
    master[f'{col}_sum8'] = master.groupby('corridor')[col].transform(
        lambda s: s.rolling(window=8, min_periods=1).sum()
    )

print(f'Added {2 * len(BINARY_FEATURES)} binary rolling-sum columns')
print(f'Master shape now: {master.shape}')

Added 4 binary rolling-sum columns
Master shape now: (6630, 74)


### Lag features

Each lag tells the model "what did this feature look like N weeks before week T?". Since the model predicts tariff events in the future, lagged features at the right horizon become the predictive signal.

| Suffix | Weeks | Approx days | Maps to |
| --- | --- | --- | --- |
| `_lag4` | 4 | ~28 | 30-day prediction horizon |
| `_lag8` | 8 | ~56 | 60-day prediction horizon |
| `_lag13` | 13 | ~91 | 90-day prediction horizon |

Lags are computed **per corridor** to avoid leakage across pairs. The first 4 / 8 / 13 rows of each corridor's series will have `NaN` lag values - this is the warm-up period and is expected. Modelling code should drop or mask these rows rather than impute them.

In [8]:
# Cell 8: Lag features at T-4, T-8, T-13 weeks

LAG_FEATURES = [
    'all_avg_tone',
    'hostility_ratio_norm',
    'tension_score',
    'log_total_articles',
    'log_trade_articles',
    'log_trade_event_count',
    'trade_event_count',
    'trade_event_flag',
    'icews_avg_intensity',
]
LAG_FEATURES = [c for c in LAG_FEATURES if c in master.columns]
LAGS = [4, 8, 13]

for col in LAG_FEATURES:
    for lag in LAGS:
        master[f'{col}_lag{lag}'] = master.groupby('corridor')[col].shift(lag)

print(f'Added {len(LAG_FEATURES) * len(LAGS)} lag columns '
      f'({len(LAG_FEATURES)} features x {len(LAGS)} lags)')
print(f'Master shape now: {master.shape}')

Added 27 lag columns (9 features x 3 lags)
Master shape now: (6630, 101)


In [9]:
# Cell 9: Final audit and save

master = master.sort_values(['corridor', 'week_start']).reset_index(drop=True)

identifiers = ['corridor', 'week_start', 'gdelt_confidence']
ma_cols   = [c for c in master.columns if c.endswith(('_ma4', '_ma8'))]
std_cols  = [c for c in master.columns if c.endswith('_std4')]
sum_cols  = [c for c in master.columns if c.endswith(('_sum4', '_sum8'))]
lag_cols  = [c for c in master.columns if '_lag' in c]
engineered = set(ma_cols + std_cols + sum_cols + lag_cols)
raw_cols  = [c for c in master.columns
             if c not in identifiers and c not in engineered]

print(f'Final feature matrix: {master.shape}')
print(f'  Rows:    {len(master):,} '
      f'({master["corridor"].nunique()} corridors x '
      f'{master["week_start"].nunique()} weeks)')
print(f'  Columns: {master.shape[1]}')
print()
print('=== Column composition ===')
print(f'  Identifiers:     {len(identifiers):>3}')
print(f'  Raw features:    {len(raw_cols):>3}')
print(f'  Rolling means:   {len(ma_cols):>3}')
print(f'  Rolling std:     {len(std_cols):>3}')
print(f'  Binary sums:     {len(sum_cols):>3}')
print(f'  Lag features:    {len(lag_cols):>3}')
print()

# Missing-value audit
print('=== Top 10 columns by missing-value count ===')
nulls = master.isnull().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False).head(10)
print(nulls.to_string() if len(nulls) else '(no nulls)')
print()
print('Leading NaNs in lag/rolling columns are expected - they are the warm-up')
print('period at the start of each corridor series. Modelling code should drop')
print('or mask these rows rather than impute them.')
print()

# Per-corridor row-count check
rows_per_corridor = master.groupby('corridor').size()
print('=== Rows per corridor (should be equal) ===')
print(rows_per_corridor.to_string())

# Save
master.to_csv('../outputs/features_master.csv', index=False)
master.to_parquet('../outputs/features_master.parquet', index=False)
print()
print('Saved: outputs/features_master.csv')
print('Saved: outputs/features_master.parquet')

Final feature matrix: (6630, 101)
  Rows:    6,630 (13 corridors x 510 weeks)
  Columns: 101

=== Column composition ===
  Identifiers:       3
  Raw features:     25
  Rolling means:    28
  Rolling std:      14
  Binary sums:       4
  Lag features:     27

=== Top 10 columns by missing-value count ===
icews_avg_intensity_lag13      188
log_trade_articles_lag13       169
all_avg_tone_lag13             169
tension_score_lag13            169
hostility_ratio_norm_lag13     169
log_total_articles_lag13       169
log_trade_event_count_lag13    169
trade_event_count_lag13        169
trade_event_flag_lag13         169
icews_avg_intensity_lag8       123

Leading NaNs in lag/rolling columns are expected - they are the warm-up
period at the start of each corridor series. Modelling code should drop
or mask these rows rather than impute them.

=== Rows per corridor (should be equal) ===
corridor
BRA_DEU    510
BRA_IND    510
BRA_USA    510
CAN_USA    510
CHN_DEU    510
CHN_IND    510
CHN_JPN    

### What's next - Stage 4

The feature matrix is the input to **Stage 4 (NVS Data Integration)** which will:

1. Join NVS duty rate records onto the weekly spine.
2. Define tariff event labels from duty rate changes.
3. Add HS code dimension (3507, 3002, 2309 - the NVS-relevant chapters).
4. Filter to the corridors where NVS has actual trade exposure (volume-weighted).
5. Hand off to the modelling layer.

Until the NVS data arrives, this matrix can be used for:

- **Sanity checks** on feature distributions and corridor coverage.
- **Prototyping** the modelling code path with GDELT trade-CAMEO events as a placeholder target.
- **Identifying** which corridor/feature combinations have enough non-null data to support modelling at each prediction horizon.

All identifiers, raw features, rolling stats, and lags carry through the file. The `gdelt_confidence` column is preserved per corridor so the modelling layer can weight or filter rows by validation quality without recomputing it.